# Do models register patient mix-ups internally?

A weekend pilot. Runs the whole study on a free Colab GPU in about an hour.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

The cells run the steps in order. Read the pilot output (step 2) before running anything
after it. If the model never flags anything, or flags everything, the later steps can't
tell you much, and it's better to know that now.

## 0. Get the code

If the repo is private, add a GitHub token with read access as a Colab secret named `GITHUB_TOKEN` (key icon on the left). If it's public, the token isn't needed.

In [ ]:
import os, subprocess
REPO = "clblanc0/evidence-probe"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = f"https://{token}@github.com/{REPO}.git"
except Exception:
    url = f"https://github.com/{REPO}.git"
if not os.path.exists("evidence-probe"):
    subprocess.run(["git", "clone", "-q", url], check=True)
%cd evidence-probe
!pip install -q -e ".[model,dev]"
!git log --oneline -1

## 1. Check the data and the text-only controls

Runs in seconds and doesn't touch the model. The two text baselines should sit near 0.5. They're the floor the probe has to beat.

In [ ]:
!python -m evidence_probe.generate --out data/charts.jsonl
!python -m evidence_probe.baselines data/charts.jsonl
!python -m pytest -q

## 2. Pilot: does the model ever flag anything?

About 5 minutes. Two prompts: `neutral` just asks for a chart entry, the way an agent would be asked to file. `check` explicitly asks the model to confirm identity first.

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
for variant in ["neutral", "check"]:
    !python -m evidence_probe.behavior data/charts.jsonl --model {MODEL} --variant {variant} --limit-bases 12 --out results/pilot_{variant}.jsonl

In [ ]:
# Read real responses before trusting any rate. Edit the scorer in behavior.py if it misreads them.
import json
for line in open("results/pilot_neutral.jsonl").readlines()[:12]:
    r = json.loads(line)
    print(f"[{r['condition']:11}] flagged={r['flagged']!s:5} | {r['response'][:160]!r}")

**Go or no-go.** Continue if, under at least one prompt:
- the model flags some tampered rows, but not nearly all rows, clean ones included, and
- reading the responses, the scorer's `flagged` matches what you would say.

If the model flags nothing under either prompt, that's still a finding and the probe is still worth running: it asks whether the activations register what the behavior misses.

## 3. Record activations

About 10 minutes. One forward pass per chart, saving every layer at two points: the end of the new visit, and the final prompt token.

In [ ]:
VARIANT = "neutral"  # Use the prompt you'll report behavior for.
!python -m evidence_probe.extract data/charts.jsonl --model {MODEL} --variant {VARIANT} --out activations/{VARIANT}.npz

## 4. Probes: accuracy by layer, controls, held-out tampering types

About 10–20 minutes, most of it the shuffled-label null. This makes figure 1.

In [ ]:
for position in ["visit_end", "final"]:
    !python -m evidence_probe.probes data/charts.jsonl activations/{VARIANT}.npz --position {position} --out-dir results --n-perm 100

In [ ]:
from IPython.display import Image, display
display(Image("results/fig1_auc_by_layer_visit_end.png"))
display(Image("results/fig1_auc_by_layer_final.png"))

**Choose the layer now**, from figure 1, before looking at behavior. Write down why. The gap analysis uses it.

In [ ]:
LAYER = None  # Set from figure 1, e.g. the peak of the blue line at visit_end.
assert LAYER is not None, "Choose the layer from figure 1 first." 

## 5. Behavior on all 540 charts

About 15–25 minutes per prompt.

In [ ]:
!python -m evidence_probe.behavior data/charts.jsonl --model {MODEL} --variant {VARIANT} --out results/behavior_{VARIANT}.jsonl --export-blind results/hand_code_{VARIANT}.csv

## 6. Inside vs behavior

This makes figure 2: for each condition, how often the probe caught it, how often the model flagged it, and how often a simple string rule would have.

In [ ]:
!python -m evidence_probe.gap results/behavior_{VARIANT}.jsonl results/probe_visit_end_oof.npz --layer {LAYER} --out-dir results
display(Image(f"results/fig2_inside_vs_behavior_{VARIANT}.png"))

## 7. Save the results

Activations are large and can be regenerated, so only `results/` is kept. Download it, or copy it to Drive.

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")

## Then, by hand

1. Open `results/hand_code_neutral.csv`. Code each response 1 (flagged an identity problem) or 0 **without looking up its condition**. Compare with the scorer and report agreement.
2. Fill in the README's results and limitations.